<a href="https://colab.research.google.com/github/ARPITSHARMA57/patient-recovery-analysis-hcl/blob/main/patient%20recovery%20and%20analyses%20code%20.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
import os
import pandas as pd

file_path = "patient_recovery_dataset.csv"

if not os.path.exists(file_path):
  from google.colab import files

  uploaded = files.upload()  # Choose patient_recovery_dataset.csv
  file_path = next(iter(uploaded))

df = pd.read_csv(file_path)


# 2. Clean missing values and duplicate rows using a function and lists
def clean_data(data):
  data = data.copy()
  print("Original rows:", len(data))

  data = data.drop_duplicates()
  print("Rows after removing duplicates:", len(data))

  num_cols = [
      "Age",
      "Length_of_Stay_Days",
      "Bilirubin",
      "Albumin",
      "Recovery_Score",
  ]
  cat_cols = ["Sex", "Department", "Admission_Type", "Recovery_Status"]

  # Fill missing numbers with the median
  for col in num_cols:
    if col in data.columns:
      data[col] = pd.to_numeric(data[col], errors="coerce")
      median_value = data[col].median()
      data[col] = data[col].fillna(median_value)

  # Fill missing text values with "Unknown"
  for col in cat_cols:
    if col in data.columns:
      data[col] = data[col].fillna("Unknown")

  return data


# 3. Add Stay_Group and Risk_Level using simple loops and if/else conditions
def add_labels(data):
  data = data.copy()
  stay_groups = []
  risk_levels = []

  for i in range(len(data)):
    stay = data.iloc[i]["Length_of_Stay_Days"]
    bili = data.iloc[i]["Bilirubin"]
    alb = data.iloc[i]["Albumin"]

    # Stay duration group
    if stay <= 6:
      stay_groups.append("Short")
    elif stay <= 12:
      stay_groups.append("Medium")
    else:
      stay_groups.append("Long")

    # Simple risk score from Bilirubin, Albumin, and Stay
    risk_points = 0
    if bili >= 2.5:
      risk_points = risk_points + 1
    if alb < 3.2:
      risk_points = risk_points + 1
    if stay >= 14:
      risk_points = risk_points + 1

    if risk_points >= 2:
      risk_levels.append("High Risk")
    elif risk_points == 1:
      risk_levels.append("Moderate Risk")
    else:
      risk_levels.append("Low Risk")

  data["Stay_Group"] = stay_groups
  data["Risk_Level"] = risk_levels
  return data


# 4. Print summary using a dictionary, groupby averages, and a set
def show_summary(data):
  # Count recovery statuses using a dictionary
  status_count = {}
  for status in data["Recovery_Status"]:
    if status in status_count:
      status_count[status] = status_count[status] + 1
    else:
      status_count[status] = 1

  print("\nRecovery Status Counts:")
  for key, value in status_count.items():
    print(f"  {key}: {value}")


  # Average values grouped by Recovery_Status
  metric_cols = [
      "Length_of_Stay_Days",
      "Bilirubin",
      "Albumin",
      "Recovery_Score",
  ]
  print("\nAverages by Recovery Status:")
  summary_table = data.groupby("Recovery_Status")[metric_cols].mean().round(2)
  print(summary_table)

  # Single line showing overall total average across all patients
  total_avg = data[metric_cols].mean()
  print(
      "Total Avg (All Patients) -> "
      f"Length_of_Stay_Days: {total_avg['Length_of_Stay_Days']:.2f} | "
      f"Bilirubin: {total_avg['Bilirubin']:.2f} | "
      f"Albumin: {total_avg['Albumin']:.2f} | "
      f"Recovery_Score: {total_avg['Recovery_Score']:.2f}"
  )
  # Find overlap using Python sets
  high_risk_patients = set(data[data["Risk_Level"] == "High Risk"]["Patient_ID"])
  long_stay_patients = set(data[data["Stay_Group"] == "Long"]["Patient_ID"])
  both_patients = high_risk_patients.intersection(long_stay_patients)

  print("\nSet Comparison:")
  print("  High Risk patients:", len(high_risk_patients))
  print("  Long Stay patients:", len(long_stay_patients))
  print("  Both High Risk & Long Stay:", len(both_patients))

  # Save cleaned file and simple text report
  data.to_csv("cleaned_patient_data.csv", index=False)
  with open("summary.txt", "w") as f:
    f.write(f"Total unique patients: {len(data)}\n")
    f.write(f"High Risk patients: {len(high_risk_patients)}\n")
    f.write(f"Long Stay patients: {len(long_stay_patients)}\n")
    f.write(f"Both High Risk & Long Stay: {len(both_patients)}\n")
  print("\nSaved 'cleaned_patient_data.csv' and 'summary.txt'.")


# Run the steps one by one
cleaned_df = clean_data(df)
final_df = add_labels(cleaned_df)
show_summary(final_df)

Original rows: 121
Rows after removing duplicates: 120

Recovery Status Counts:
  Recovered: 95
  Extended Care: 17
  Readmitted (30d): 8

Averages by Recovery Status:
                  Length_of_Stay_Days  Bilirubin  Albumin  Recovery_Score
Recovery_Status                                                          
Extended Care                   15.59       3.02     2.92           62.29
Readmitted (30d)                 9.38       3.50     2.88           66.15
Recovered                        8.32       2.24     3.66           86.04
Total Avg (All Patients) -> Length_of_Stay_Days: 9.42 | Bilirubin: 2.43 | Albumin: 3.50 | Recovery_Score: 81.35

Set Comparison:
  High Risk patients: 33
  Long Stay patients: 31
  Both High Risk & Long Stay: 19

Saved 'cleaned_patient_data.csv' and 'summary.txt'.
